# Desarrollo de un Sistema de Generación Aumentada por Recuperación (RAG) para la Consulta y Auditoría de Normativas Internacionales de Inteligencia Artificial.
## Sistema RAG

**Autor:** Eduard David Apolo Gallardo  
**Máster en Deep Learning — UPM**

---

Esta fase construye el pipeline RAG completo usando el modelo generativo LLaMA 3.1 8B cuantizado a 4 bits, inferencia local via Ollama.

---
## Importaciones y Carga de corpus

In [2]:
import os
import re
import json
import time
import random
import logging
import numpy as np
import torch
import pickle
import unicodedata
from pathlib import Path
from langdetect import detect, DetectorFactory
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langdetect.lang_detect_exception import LangDetectException
from langchain_community.docstore.document import Document
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
import gc

def purgar_vram_gpu(variables_a_eliminar: list = None) -> None:
    if variables_a_eliminar is not None:
        for var in variables_a_eliminar:
            try:
                del var
            except NameError:
                pass
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        
        print("Purga de VRAM completada. Memoria reservada actual: "
              f"{torch.cuda.memory_reserved(0) / (1024**2):.2f} MB")
    else:
        print("No se detectó un dispositivo CUDA activo.")


purgar_vram_gpu()
os.environ["HF_HUB_DISABLE_XET"] = "1"

# SEMILLAS
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
DetectorFactory.seed = 42

print("Semillas establecidas para reproducibilidad.")

Purga de VRAM completada. Memoria reservada actual: 482.00 MB
Semillas establecidas para reproducibilidad.


C:\Users\a3dua\AppData\Local\Temp\ipykernel_18368\2575968337.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.document import Document


In [3]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

BASE_DIR           = Path(r"E:\Proyectos_IA\Deep_Learning\TFM")
DATA_DIR           = BASE_DIR / "Data"
VECTOR_DIR         = BASE_DIR / "VectorDB"
LOG_DIR            = BASE_DIR / "Logs"
CHROMA_BASELINE_DIR = VECTOR_DIR / "chroma_baseline"
CHROMA_BGE_M3_DIR  = VECTOR_DIR / "chroma_bge_m3"
GROUND_TRUTH_DIR   = BASE_DIR / "GroundTruth"
GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "fase3_rag.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Device: {device}")

# CARGA DEL CORPUS 
with open(BASE_DIR / "fase2_estado.json", "r", encoding="utf-8") as f:
    estado_fase2 = json.load(f)

logger.info("Estado de Fase 2 cargado correctamente.")

with open(DATA_DIR / "corpus_tfm_procesado.json", "r", encoding="utf-8") as f:
    corpus_raw = json.load(f)

fragmentos = [
    Document(page_content=item["contenido"], metadata=item["metadatos"])
    for item in corpus_raw
]

logger.info(f"Corpus cargado: {len(fragmentos)} fragmentos")

# CARGA DE MODELOS DE EMBEDDINGS
import gc

# Primero limpiar cualquier estado GPU previo
purgar_vram_gpu()

logger.info("Cargando modelo baseline MiniLM-L12...")
modelo_baseline = SentenceTransformer(
    estado_fase2["modelos"]["baseline"]["id"],
    device=device
)
# Mover a CPU tras la carga para liberar VRAM antes de cargar bge-m3
modelo_baseline.to("cpu")
logger.info("Modelo baseline cargado y movido a CPU.")

# Liberación explícita de VRAM antes de cargar bge-m3
purgar_vram_gpu()


logger.info("Cargando modelo bge-m3...")
modelo_bge_m3 = SentenceTransformer(
    estado_fase2["modelos"]["bge_m3"]["id"],
    device=device,
    model_kwargs={"torch_dtype": torch.float16}
)
logger.info("Modelo bge-m3 cargado en GPU.")

# CHROMADB
cliente_baseline = chromadb.PersistentClient(
    path=str(CHROMA_BASELINE_DIR),
    settings=Settings(anonymized_telemetry=False)
)
cliente_bge_m3 = chromadb.PersistentClient(
    path=str(CHROMA_BGE_M3_DIR),
    settings=Settings(anonymized_telemetry=False)
)

coleccion_baseline = cliente_baseline.get_collection("ai_act_baseline")
coleccion_bge_m3   = cliente_bge_m3.get_collection("ai_act_bge_m3")

logger.info(f"ChromaDB baseline: {coleccion_baseline.count()} vectores")
logger.info(f"ChromaDB bge-m3  : {coleccion_bge_m3.count()} vectores")

mapa_chunk_id = {f.metadata["chunk_id"]: i for i, f in enumerate(fragmentos)}

# CARGA DEL RECUPERADOR LÉXICO (BM25)
RUTA_BM25 = VECTOR_DIR / "indice_bm25.pkl"
STOPWORDS_ES = {
    "de", "la", "el", "en", "y", "a", "los", "las", "se", "del",
    "que", "un", "una", "con", "por", "para", "es", "al", "lo",
    "su", "sus", "o", "le", "como", "más", "pero", "sus", "no",
    "ha", "me", "si", "ya", "sea", "así", "este", "esta", "estos",
    "estas", "ese", "esa", "esos", "esas", "ser", "han", "será",
    "cuáles", "cuales", "son", "según", "segun", "qué", "que",
    "cómo", "como", "dónde", "donde", "cuándo", "cuando",
    "cuál", "cual", "quién", "quien", "quiénes", "quienes",
    "hay", "debe", "deben", "puede", "pueden", "tiene", "tienen",
    "dicho", "dichos", "dicha", "dichas", "mismo", "misma",
    "tal", "tales", "cada", "entre", "sobre", "ante", "bajo",
    "tras", "sin", "hasta", "desde", "hacia", "durante",
    "mediante", "contra", "salvo", "excepto", "sino",
    "también", "tampoco", "además", "incluso", "aunque",
    "mientras", "porque", "pues", "sino", "ni",
}

# Funciones de normalización y tokenización
def normalizar_acentos(texto: str) -> str:
    """Elimina diacríticos para homogeneizar el texto."""
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

def tokenizar_juridico(texto: str) -> list[str]:
    """Tokeniza la consulta del usuario con las mismas reglas del corpus."""
    texto = normalizar_acentos(texto)
    texto = texto.lower()
    texto = re.sub(r"[^\w\s\-]", " ", texto)
    tokens = [t for t in texto.split() if t not in STOPWORDS_ES and len(t) >= 3]
    return tokens

# Cargar el índice desde el disco
logger.info("Cargando recuperador léxico BM25..")
try:
    with open(RUTA_BM25, "rb") as f:
        bm25_data = pickle.load(f)
    indice_bm25 = bm25_data["indice"]
    logger.info("Índice BM25 cargado correctamente en memoria.")
    print(f"Índice BM25 operativo.")
    
except FileNotFoundError:
    logger.error(f"No se encontró el archivo {RUTA_BM25}. Debes ejecutar la Fase 2 primero.")
    print(f"[ERROR] Índice BM25 no encontrado en la ruta: {RUTA_BM25}")

print("Estado de carga finalizado correctamente.")
print(f"  Fragmentos    : {len(fragmentos)}")
print(f"  ChromaDB base : {coleccion_baseline.count()} vectores")
print(f"  ChromaDB bge  : {coleccion_bge_m3.count()} vectores")

2026-06-13 15:08:18,500 [INFO] Device: cuda
2026-06-13 15:08:18,514 [INFO] Estado de Fase 2 cargado correctamente.
2026-06-13 15:08:18,619 [INFO] Corpus cargado: 5192 fragmentos
2026-06-13 15:08:18,756 [INFO] Cargando modelo baseline MiniLM-L12...


Purga de VRAM completada. Memoria reservada actual: 482.00 MB


2026-06-13 15:08:19,199 [INFO] Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.
2026-06-13 15:08:23,361 [INFO] Modelo baseline cargado y movido a CPU.
2026-06-13 15:08:23,548 [INFO] Cargando modelo bge-m3...


Purga de VRAM completada. Memoria reservada actual: 482.00 MB


2026-06-13 15:08:23,975 [INFO] Loading SentenceTransformer model from BAAI/bge-m3.
2026-06-13 15:08:29,264 [INFO] Modelo bge-m3 cargado en GPU.
2026-06-13 15:08:29,430 [INFO] ChromaDB baseline: 5192 vectores
2026-06-13 15:08:29,518 [INFO] ChromaDB bge-m3  : 5192 vectores
2026-06-13 15:08:29,520 [INFO] Cargando recuperador léxico BM25..
2026-06-13 15:08:29,613 [INFO] Índice BM25 cargado correctamente en memoria.


Índice BM25 operativo.
Estado de carga finalizado correctamente.
  Fragmentos    : 5192
  ChromaDB base : 5192 vectores
  ChromaDB bge  : 5192 vectores


---
## Conexión al modelo LLaMA 3.1 8B via Ollama

In [4]:
MODELO_LLM_ID  = "llama3.1:8b"
OLLAMA_BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=MODELO_LLM_ID,
    base_url=OLLAMA_BASE_URL,
    temperature=0.0,      # Determinismo
    num_predict=1024,     # Longitud máxima de respuesta en tokens
    num_ctx=4096,         # Ventana de contexto
)

logger.info(f"Modelo LLM configurado: {MODELO_LLM_ID} (temperatura=0.0)")

# VERIFICACIÓN DE CONEXIÓN
print("Verificando conexión con Ollama...")
t0 = time.time()

respuesta_test = llm.invoke([
    SystemMessage(content="Eres un asistente de auditoría legal. Responde siempre en español."),
    HumanMessage(content="Confirma que estás operativo respondiendo con: 'Sistema RAG operativo.'")
])

t_respuesta = time.time() - t0
logger.info(f"Conexión verificada en {t_respuesta:.1f}s")

print(f"\n[OK] Modelo operativo — Tiempo de respuesta: {t_respuesta:.1f}s")
print(f"Respuesta del modelo: {respuesta_test.content}")

2026-06-13 15:08:55,202 [INFO] Modelo LLM configurado: llama3.1:8b (temperatura=0.0)


Verificando conexión con Ollama...


2026-06-13 15:09:08,406 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:09:09,583 [INFO] Conexión verificada en 14.4s



[OK] Modelo operativo — Tiempo de respuesta: 14.4s
Respuesta del modelo: Sistema RAG operativo. ¿En qué puedo ayudarte?


---
## Sistema de prompts

Se diseñan el system prompt y el user prompt template del sistema RAG.
El system prompt restringe explícitamente al modelo a fundamentar sus
respuestas exclusivamente en el contexto normativo recuperado, prohibiendo
el uso del conocimiento paramétrico para afirmaciones normativas.

In [5]:
# SYSTEM PROMPT
SYSTEM_PROMPT = """Eres un asistente experto en auditoría legal del Reglamento (UE) 2024/1689 \
(Ley Europea de Inteligencia Artificial, AI Act) y normativas complementarias de inteligencia \
artificial.

INSTRUCCIONES OBLIGATORIAS:
1. Responde ÚNICAMENTE en español.
2. Basa tu respuesta EXCLUSIVAMENTE en los fragmentos normativos proporcionados en el contexto.
3. NUNCA inventes artículos, obligaciones, plazos o sanciones que no aparezcan en el contexto.
4. Si el contexto no contiene información suficiente para responder, indícalo explícitamente:
   'La información proporcionada no es suficiente para responder esta consulta con certeza normativa.'
5. Cita siempre las fuentes utilizando el formato de citación establecido.
6. Mantén un tono formal y técnico-jurídico.
7. Estructura la respuesta en: respuesta directa → desarrollo → citaciones."""

# Inyección de contexto y consulta
USER_PROMPT_TEMPLATE = """CONTEXTO NORMATIVO RECUPERADO:
{contexto}

---
CONSULTA DE AUDITORÍA:
{consulta}

---
Proporciona una respuesta fundamentada basándote exclusivamente en el contexto normativo \
anterior. Incluye las citaciones correspondientes al final de la respuesta."""


print("System prompt:")
print("-" * 55)
print(SYSTEM_PROMPT)
print("-" * 55)
print(f"\nLongitud system prompt : {len(SYSTEM_PROMPT)} chars")
print(f"Longitud user template : {len(USER_PROMPT_TEMPLATE)} chars")

System prompt:
-------------------------------------------------------
Eres un asistente experto en auditoría legal del Reglamento (UE) 2024/1689 (Ley Europea de Inteligencia Artificial, AI Act) y normativas complementarias de inteligencia artificial.

INSTRUCCIONES OBLIGATORIAS:
1. Responde ÚNICAMENTE en español.
2. Basa tu respuesta EXCLUSIVAMENTE en los fragmentos normativos proporcionados en el contexto.
3. NUNCA inventes artículos, obligaciones, plazos o sanciones que no aparezcan en el contexto.
4. Si el contexto no contiene información suficiente para responder, indícalo explícitamente:
   'La información proporcionada no es suficiente para responder esta consulta con certeza normativa.'
5. Cita siempre las fuentes utilizando el formato de citación establecido.
6. Mantén un tono formal y técnico-jurídico.
7. Estructura la respuesta en: respuesta directa → desarrollo → citaciones.
-------------------------------------------------------

Longitud system prompt : 828 chars
Longitud

---
## Formateador de contexto con citación estructurada

Se implementa el sistema de formateo del contexto recuperado con
citaciones estructuradas. El formato distingue entre:
- Fuente principal: Reglamento (UE) 2024/1689 (AI Act)
- Fuentes complementarias: AESIA, AEPD, NIST, ISO, etc.

In [6]:
NOMBRES_FORMALES_FUENTE = {
    "Reglamento (UE) 20241689 (AI Act)" : "Reglamento (UE) 2024/1689 — AI Act",
    "AESIA"                              : "Agencia Española de Supervisión de IA (AESIA)",
    "AEPD EIPD"                          : "Agencia Española de Protección de Datos (AEPD)",
    "NIST AI RMF"                        : "NIST AI Risk Management Framework 1.0",
    "ISO IEC 42001"                      : "Estándar ISO/IEC 42001:2023",
    "AI HLEG"                            : "AI HLEG — Comisión Europea",
    "OCDE"                               : "Principios de IA de la OCDE",
}


def obtener_nombre_formal(categoria: str) -> str:
    """Retorna el nombre formal de citación para una categoría de fuente."""
    for clave, nombre in NOMBRES_FORMALES_FUENTE.items():
        if clave.lower() in categoria.lower():
            return nombre
    return categoria


def es_fuente_principal(categoria: str) -> bool:
    """Determina si el fragmento proviene del AI Act (fuente principal)."""
    return "20241689" in categoria or "AI Act" in categoria


def formatear_contexto_con_citas(resultados: list[dict]) -> tuple[str, list[dict]]:
    bloques_contexto = []
    referencias = []

    for i, resultado in enumerate(resultados):
        meta      = resultado["metadatos"]
        contenido = resultado["contenido"]
        n_ref     = i + 1

        # Extracción de metadatos de trazabilidad
        categoria    = meta.get("categoria_fuente", meta.get("source", "Fuente desconocida"))
        articulos    = meta.get("articulos", "")
        pagina       = meta.get("page", meta.get("pagina", "N/D"))
        nombre_formal = obtener_nombre_formal(str(categoria))

        # Construcción del bloque de contexto con marcador de referencia
        arts_str = f" — Artículo(s): {articulos}" if articulos else ""
        bloque = (
            f"[Fuente {n_ref}] {nombre_formal}{arts_str}\n"
            f"{contenido.strip()}"
        )
        bloques_contexto.append(bloque)

        # Construcción de la referencia para citación al pie
        referencias.append({
            "n_ref"        : n_ref,
            "nombre_formal": nombre_formal,
            "articulos"    : articulos,
            "pagina"       : pagina,
            "categoria"    : categoria,
            "es_principal" : es_fuente_principal(str(categoria))
        })

    contexto_formateado = "\n\n".join(bloques_contexto)
    return contexto_formateado, referencias


def formatear_citaciones(referencias: list[dict]) -> str:
    """
    Genera el bloque de citaciones estructuradas para añadir
    al final de cada respuesta del sistema RAG.

    Formato: 'Según el Artículo X del [Fuente]: ...'
    """
    lineas = ["\n--- FUENTES NORMATIVAS CONSULTADAS ---"]

    # Primero las fuentes principales (AI Act)
    principales = [r for r in referencias if r["es_principal"]]
    complementarias = [r for r in referencias if not r["es_principal"]]

    for ref in principales:
        arts = ref["articulos"]
        if arts:
            lineas.append(
                f"[{ref['n_ref']}] {ref['nombre_formal']} — "
                f"Artículo(s): {arts}"
            )
        else:
            lineas.append(f"[{ref['n_ref']}] {ref['nombre_formal']}")

    if complementarias:
        lineas.append("\nFuentes complementarias:")
        for ref in complementarias:
            pag = f" (pág. {ref['pagina']})" if ref["pagina"] not in ["", "N/D"] else ""
            lineas.append(f"[{ref['n_ref']}] {ref['nombre_formal']}{pag}")

    return "\n".join(lineas)


print("Sistema de citación estructurada configurado.")
print(f"     Fuentes formales registradas: {len(NOMBRES_FORMALES_FUENTE)}")

Sistema de citación estructurada configurado.
     Fuentes formales registradas: 7


---
## Pipeline RAG completo

Se integran todos los componentes en la función principal del sistema RAG:
1. Recuperación híbrida (BM25 + bge-m3 + RRF) con filtro de idioma
2. Formateo del contexto con citaciones
3. Construcción del prompt
4. Generación de respuesta con LLaMA 3.1 8B
5. Ensamblado de respuesta + citaciones

In [7]:
def busqueda_semantica_RAG(
    consulta: str,
    modelo: SentenceTransformer,
    coleccion: chromadb.Collection,
    k: int = 10,
    filtrar_idioma: str = "es"
) -> list[dict]:
    vector = modelo.encode(consulta, normalize_embeddings=True)

    kwargs = {
        "query_embeddings" : [vector.tolist()],
        "n_results"        : k,
        "include"          : ["documents", "metadatas", "distances"]
    }
    # Filtro de idioma aplicado como condición de metadatos en ChromaDB
    if filtrar_idioma:
        kwargs["where"] = {"idioma": filtrar_idioma}

    resultados = coleccion.query(**kwargs)

    return [
        {
            "contenido" : resultados["documents"][0][i],
            "metadatos" : resultados["metadatas"][0][i],
            "distancia" : resultados["distances"][0][i],
            "indice"    : mapa_chunk_id.get(
                int(resultados["metadatas"][0][i].get("chunk_id", -1)), -1
            ),
            "ranking"   : i + 1
        }
        for i in range(len(resultados["documents"][0]))
    ]


def busqueda_bm25_RAG(
    consulta: str,
    indice: BM25Okapi,
    fragmentos: list[Document],
    k: int = 10,
    filtrar_idioma: str = "es"
) -> list[dict]:
    """Búsqueda BM25 con filtro opcional de idioma."""
    tokens = tokenizar_juridico(consulta)
    puntuaciones = indice.get_scores(tokens)

    indices_ordenados = np.argsort(puntuaciones)[::-1]

    resultados = []
    for idx in indices_ordenados:
        if len(resultados) >= k:
            break
        f = fragmentos[idx]
        # Filtro de idioma sobre metadatos del fragmento
        if filtrar_idioma and f.metadata.get("idioma", "") != filtrar_idioma:
            continue
        resultados.append({
            "contenido" : f.page_content,
            "metadatos" : f.metadata,
            "score"     : float(puntuaciones[idx]),
            "indice"    : int(idx),
            "ranking"   : len(resultados) + 1
        })

    return resultados


def reciprocal_rank_fusion(
    lista_rankings: list[list[dict]],
    fragmentos: list[Document],
    k_rrf: int = 60,
    top_n: int = 5,
    boost_ai_act: bool = True
) -> list[dict]:
    """Fusión RRF de múltiples rankings."""
    puntuaciones = {}

    for ranking in lista_rankings:
        for pos, resultado in enumerate(ranking):
            idx = resultado["indice"]
            score_base = 1.0 / (k_rrf + pos + 1)
            # Boost del 20% a fragmentos del AI Act principal
            if boost_ai_act:
                cat = fragmentos[idx].metadata.get("categoria_fuente", "")
                if "20241689" in str(cat):
                    score_base *= 1.2
            puntuaciones[idx] = puntuaciones.get(idx, 0.0) + score_base

    indices_top = sorted(puntuaciones, key=lambda x: puntuaciones[x], reverse=True)[:top_n]

    return [
        {
            "contenido" : fragmentos[idx].page_content,
            "metadatos" : fragmentos[idx].metadata,
            "score_rrf" : puntuaciones[idx],
            "indice"    : idx,
            "ranking"   : i + 1
        }
        for i, idx in enumerate(indices_top)
    ]

# PIPELINE RAG COMPLETO
def consulta_rag(
    consulta: str,
    k_candidatos: int = 20,
    k_final: int = 5,
    filtrar_idioma: str = "es",
    verbose: bool = True
) -> dict:
    t_inicio = time.time()

    # RECUPERACIÓN HÍBRIDA (BM25 + bge-m3 + RRF)
    t0 = time.time()
    res_sem = busqueda_semantica_RAG(
        consulta, modelo_bge_m3, coleccion_bge_m3, k_candidatos, filtrar_idioma
    )
    res_bm25 = busqueda_bm25_RAG(
        consulta, indice_bm25, fragmentos, k_candidatos, filtrar_idioma
    )
    fragmentos_recuperados = reciprocal_rank_fusion(
        [res_sem, res_bm25], fragmentos, k_rrf=60, top_n=k_final
    )
    t_recuperacion = time.time() - t0

    if verbose:
        print(f"\n[Recuperación] {len(fragmentos_recuperados)} fragmentos en {t_recuperacion:.2f}s")
        for r in fragmentos_recuperados:
            cat = r['metadatos'].get('categoria_fuente', r['metadatos'].get('source', 'N/D'))
            arts = r['metadatos'].get('articulos', '')
            print(f"  [{r['ranking']}] Score RRF: {r['score_rrf']:.5f} | "
                  f"Fuente: {str(cat)[:40]} | Arts: {arts}")

    # FORMATEO DEL CONTEXTO CON CITACIONES
    contexto, referencias = formatear_contexto_con_citas(fragmentos_recuperados)

    # CONSTRUCCIÓN DEL PROMPT
    user_prompt = USER_PROMPT_TEMPLATE.format(
        contexto=contexto,
        consulta=consulta
    )

    # GENERACIÓN DE RESPUESTA
    t0 = time.time()
    respuesta_llm = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ])
    t_generacion = time.time() - t0

    if verbose:
        print(f"[Generación]   Respuesta generada en {t_generacion:.2f}s")

    # ENSAMBLADO DE RESPUESTA FINAL CON CITACIONES
    citaciones_str = formatear_citaciones(referencias)
    respuesta_final = respuesta_llm.content + "\n" + citaciones_str

    t_total = time.time() - t_inicio

    return {
        "consulta"          : consulta,
        "respuesta"         : respuesta_final,
        "respuesta_raw"     : respuesta_llm.content,
        "citaciones"        : referencias,
        "fragmentos"        : fragmentos_recuperados,
        "contexto"          : contexto,
        "tiempo_recuperacion": t_recuperacion,
        "tiempo_generacion" : t_generacion,
        "tiempo_total"      : t_total
    }


print("Pipeline RAG completo configurado.")
print("     Recuperador activo : HybM (BM25 + bge-m3 + RRF)")
print("     Modelo generativo  : LLaMA 3.1 8B (Ollama, temp=0.0)")
print("     Filtro de idioma   : español (es)")
print("     Sistema de citación: activo")

Pipeline RAG completo configurado.
     Recuperador activo : HybM (BM25 + bge-m3 + RRF)
     Modelo generativo  : LLaMA 3.1 8B (Ollama, temp=0.0)
     Filtro de idioma   : español (es)
     Sistema de citación: activo


---
## Pruebas del pipeline con consultas de auditoría reales

Se ejecutan cinco consultas representativas del dominio de auditoría del AI Act
para verificar el funcionamiento completo del pipeline RAG.

In [8]:
CONSULTAS_PRUEBA = [
    "¿Cuáles son las obligaciones del proveedor de un sistema de IA de alto riesgo?",
    "¿Qué requisitos de transparencia debe cumplir un sistema de IA que interactúa con personas?",
    "¿Qué sanciones económicas establece el AI Act por incumplimiento grave?",
]

resultados_prueba = []

for i, consulta in enumerate(CONSULTAS_PRUEBA):
    print("\n" + "=" * 65)
    print(f"CONSULTA {i+1}/{len(CONSULTAS_PRUEBA)}")
    print("=" * 65)
    print(f"Pregunta: {consulta}")
    print("-" * 65)

    resultado = consulta_rag(consulta, k_candidatos=20, k_final=5, verbose=True)
    resultados_prueba.append(resultado)

    print("\nRESPUESTA:")
    print(resultado["respuesta"])
    print(f"\nTiempo total: {resultado['tiempo_total']:.1f}s")


CONSULTA 1/3
Pregunta: ¿Cuáles son las obligaciones del proveedor de un sistema de IA de alto riesgo?
-----------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[Recuperación] 5 fragmentos en 0.58s
  [1] Score RRF: 0.03227 | Fuente: AESIA | Arts: ['12']
  [2] Score RRF: 0.01846 | Fuente: Reglamento (UE) 20241689 (AI Act) | Arts: []
  [3] Score RRF: 0.01639 | Fuente: AESIA | Arts: []
  [4] Score RRF: 0.01613 | Fuente: AESIA | Arts: []
  [5] Score RRF: 0.01613 | Fuente: AESIA | Arts: []


2026-06-13 15:09:54,014 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


[Generación]   Respuesta generada en 71.32s

RESPUESTA:
Respuesta directa:
El proveedor de un sistema de IA de alto riesgo tiene varias obligaciones, entre ellas:

* Conservar los registros generados por su sistema durante un período de al menos seis meses (Fuente 1, artículo 12).
* Tomar las medidas adecuadas para garantizar la protección en el aspecto de ciberseguridad para la IA (Fuente 3, apartado 1.3).
* Establecer un nivel de precisión acorde a la finalidad prevista del sistema y establecer unas medidas adecuadas durante todo el ciclo de vida del sistema (Fuente 4).

Desarrollo:
Según el Reglamento (UE) 2024/1689, cualquier distribuidor, importador, responsable del despliegue o tercero que ponga su nombre o marca en un sistema de IA de alto riesgo previamente introducido en el mercado o puesto en servicio será considerado proveedor a efectos del presente Reglamento (Fuente 2, artículo 25). Esto significa que el proveedor tiene la responsabilidad de cumplir con las obligaciones es

...

[Generación]   Respuesta generada en 89.95s

RESPUESTA:
Respuesta directa:
Un sistema de IA que interactúa con personas debe cumplir los siguientes requisitos de transparencia:

1. Proporcionar información a todos los perfiles que interaccionan con él, asegurando su completo entendimiento de manera transparente (Fuente 2).
2. Habilitar mecanismos técnicos para mostrar dicha información de manera transparente y entendible por todos ellos, adecuando el tipo de lenguaje a su nivel de interlocución (Fuente 2).
3. Proporcionar una interfaz de usuario alineada con el nivel de interlocución de cada uno de los perfiles que interactúa con él, facilitando la siguiente información de manera fácilmente entendible, visual y textualmente en lenguaje natural (Fuente 1 y Fuente 2).
4. Proporcionar instrucciones de uso que incluyan información concisa, completa, correcta y clara que sea pertinente, accesible y comprensible para los responsables del despliegue (Fuente 3).
5. Proporcionar un conjunto de 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[Recuperación] 5 fragmentos en 0.17s
  [1] Score RRF: 0.03644 | Fuente: Reglamento (UE) 20241689 (AI Act) | Arts: []
  [2] Score RRF: 0.03126 | Fuente: AESIA | Arts: ['20']
  [3] Score RRF: 0.03125 | Fuente: AESIA | Arts: ['73', '72']
  [4] Score RRF: 0.02947 | Fuente: AESIA | Arts: []
  [5] Score RRF: 0.01967 | Fuente: Reglamento (UE) 20241689 (AI Act) | Arts: []


2026-06-13 15:12:31,304 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


[Generación]   Respuesta generada en 54.92s

RESPUESTA:
Respuesta directa: El Reglamento (UE) 2024/1689 — AI Act establece multas administrativas por incumplimiento grave, con una cuantía máxima de hasta 35 000 000 EUR o, si el infractor es una empresa, de hasta el 7 % de su volumen de negocios mundial total correspondiente al ejercicio financiero anterior.

Desarrollo: Según el artículo 99 del Reglamento (UE) 2024/1689 — AI Act, los Estados miembros deben establecer un régimen de sanciones y otras medidas de ejecución para las infracciones cometidas por operadores. Estas sanciones deben ser efectivas, proporcionadas y disuasorias, teniendo en cuenta los intereses de las pymes, incluidas las empresas emergentes, así como su viabilidad económica (artículo 99, apartado 1). Además, el incumplimiento de la prohibición de prácticas de IA establecida en el artículo 5 estará sujeto a multas administrativas de hasta 35 000 000 EUR o, si el infractor es una empresa, de hasta el 7 % de su volume

---
## Generación del conjunto de evaluación ground truth

Se genera un conjunto de pares pregunta-respuesta de referencia sobre
artículos conocidos del AI Act.

In [12]:
N_PARES_GROUND_TRUTH = 60

fragmentos_ai_act_es = [
    f for f in fragmentos
    if "AI Act" in f.metadata.get("categoria_fuente", "")
    and len(f.page_content) >= 300
]

logger.info(f"Fragmentos AI Act disponibles tras filtro: {len(fragmentos_ai_act_es)}")
print(f"Fragmentos disponibles para ground truth: {len(fragmentos_ai_act_es)}")

# Selección de la muestra
if len(fragmentos_ai_act_es) >= N_PARES_GROUND_TRUTH:
    fragmentos_seleccionados = random.sample(fragmentos_ai_act_es, N_PARES_GROUND_TRUTH)
elif len(fragmentos_ai_act_es) > 0:
    fragmentos_seleccionados = fragmentos_ai_act_es
    logger.warning(f"Solo hay {len(fragmentos_ai_act_es)} fragmentos disponibles.")
else:
    raise ValueError("Sin fragmentos.")

print(f"Fragmentos seleccionados para ground truth: {len(fragmentos_seleccionados)}")

PROMPT_GENERACION_QA = """Dado el siguiente fragmento del Reglamento (UE) 2024/1689 (AI Act), \
genera exactamente UN par pregunta-respuesta de auditoría legal.

FRAGMENTO NORMATIVO:
{fragmento}

INSTRUCCIONES:
- La pregunta debe ser una consulta de auditoría realista que alguien haría sobre este texto.
- La respuesta debe estar basada EXCLUSIVAMENTE en el fragmento proporcionado.
- Responde en español.
- Usa EXACTAMENTE este formato JSON (sin texto adicional antes o después). Si el fragmento menciona el número de artículo, ponlo en la lista; si no, déjala vacía:
{{
  "pregunta": "<pregunta de auditoría>",
  "respuesta_referencia": "<respuesta basada en el fragmento>",
  "articulos_referenciados": ["<artículo X>", "<artículo Y>"]
}}"""

# GENERACIÓN DEL GROUND TRUTH
pares_ground_truth = []
errores = 0

print(f"\nGenerando {len(fragmentos_seleccionados)} pares Q&A...")
print("(Este proceso puede tardar varios minutos dependiendo de Ollama)")

for i, fragmento in enumerate(fragmentos_seleccionados):
    try:
        prompt = PROMPT_GENERACION_QA.format(
            fragmento=fragmento.page_content[:800] 
        )

        respuesta = llm.invoke([
            SystemMessage(content="Eres un experto legal generando pares Q&A en formato JSON estricto."),
            HumanMessage(content=prompt)
        ])

        contenido = respuesta.content.strip()
        contenido = re.sub(r"^```json|```$", "", contenido, flags=re.IGNORECASE|re.MULTILINE).strip()

        par = json.loads(contenido)

        if "pregunta" in par and "respuesta_referencia" in par:
            par["chunk_id"]         = fragmento.metadata.get("chunk_id", i)
            par["fuente"]           = fragmento.metadata.get("categoria_fuente", "")
            # Usamos los artículos del LLM si los encontró, si no, los del metadato
            par["articulos_fuente"] = par.get("articulos_referenciados", fragmento.metadata.get("articulos", []))
            par["contexto"]         = fragmento.page_content
            
            pares_ground_truth.append(par)

        if (i + 1) % 10 == 0:
            print(f"  Progreso: {i+1}/{len(fragmentos_seleccionados)} "
                  f"({len(pares_ground_truth)} pares válidos)")

    except Exception as e:
        errores += 1
        logger.warning(f"Error parseando JSON en fragmento {i}: {str(e)[:50]}...")
        continue

# PERSISTENCIA DEL GROUND TRUTH
RUTA_GROUND_TRUTH = GROUND_TRUTH_DIR / "ground_truth_ai_act.json"

with open(RUTA_GROUND_TRUTH, "w", encoding="utf-8") as f:
    json.dump(pares_ground_truth, f, ensure_ascii=False, indent=2)

print(f"\n{'='*55}")
print("GROUND TRUTH GENERADO")
print(f"{'='*55}")
print(f"  Pares generados exitosamente : {len(pares_ground_truth)}")
print(f"  Errores de parseo JSON       : {errores}")
print(f"  Fichero                      : {RUTA_GROUND_TRUTH.name}")

if pares_ground_truth:
    print(f"\nEjemplo de par generado:")
    ej = pares_ground_truth[0]
    print(f"  Pregunta  : {ej['pregunta']}")
    print(f"  Respuesta : {ej['respuesta_referencia'][:150]}...")

2026-06-13 15:26:46,762 [INFO] Fragmentos AI Act disponibles tras filtro: 1978


Fragmentos disponibles para ground truth: 1978
Fragmentos seleccionados para ground truth: 60

Generando 60 pares Q&A...
(Este proceso puede tardar varios minutos dependiendo de Ollama)


2026-06-13 15:26:56,336 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:27:08,778 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:27:20,873 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:27:33,180 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:27:41,894 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:27:59,418 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:28:13,543 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:28:22,853 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:28:46,389 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:28:56,785 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


  Progreso: 10/60 (10 pares válidos)


2026-06-13 15:29:07,930 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:29:16,390 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:29:25,771 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:29:34,973 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:29:45,208 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:30:05,979 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:30:19,209 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:30:30,144 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:30:45,100 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:30:56,675 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


...

  Progreso: 40/60 (40 pares válidos)


2026-06-13 15:34:56,966 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:35:06,506 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:35:18,804 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:35:32,965 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:35:43,371 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:35:57,040 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:36:12,487 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:36:25,950 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:36:39,464 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:36:52,485 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


  Progreso: 50/60 (50 pares válidos)


2026-06-13 15:37:03,796 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:37:16,316 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:37:34,957 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:37:49,374 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:38:01,628 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:38:16,115 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:38:28,465 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:38:45,411 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:38:55,230 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:39:07,263 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


  Progreso: 60/60 (60 pares válidos)

GROUND TRUTH GENERADO
  Pares generados exitosamente : 60
  Errores de parseo JSON       : 0
  Fichero                      : ground_truth_ai_act.json

Ejemplo de par generado:
  Pregunta  : ¿Qué técnicas manipulativas o engañosas están prohibidas según este reglamento?
  Respuesta : Las técnicas que utilizan componentes subliminales, estímulos de audio, imagen o vídeo que las personas no pueden percibir, y otras técnicas manipulat...


---
## Evaluación cuantitativa

Se evalúa el sistema RAG sobre el ground truth generado mediante la implementación directa de las 4 métricas usando el LLM local:
- **Faithfulness** ≥ 0.85 — ausencia de alucinaciones
- **Answer Relevancy** ≥ 0.80 — pertinencia de la respuesta
- **Context Precision** ≥ 0.75 — calidad de la recuperación
- **Context Recall** ≥ 0.75 — cobertura de la recuperación

In [14]:
# EVALUACIÓN RAG
# Implementación directa de las 4 métricas usando el LLM local
N_EVAL = min(20, len(pares_ground_truth))
gt_eval = pares_ground_truth[:N_EVAL]

print(f"Evaluando {N_EVAL} pares Q&A...")

scores = {
    "faithfulness"      : [],
    "answer_relevancy"  : [],
    "context_precision" : [],
    "context_recall"    : [],
}

PROMPT_FAITHFULNESS = """Dado el siguiente contexto y respuesta, evalúa si la respuesta
está completamente fundamentada en el contexto (sin inventar información).
Contexto: {contexto}
Respuesta: {respuesta}
Responde SOLO con un número decimal entre 0.0 y 1.0 donde:
1.0 = completamente fundamentada, 0.0 = completamente inventada.
Responde únicamente con el número, sin texto adicional."""

PROMPT_RELEVANCY = """Dada la pregunta y la respuesta, evalúa qué tan relevante
es la respuesta para la pregunta.
Pregunta: {pregunta}
Respuesta: {respuesta}
Responde SOLO con un número decimal entre 0.0 y 1.0 donde:
1.0 = completamente relevante, 0.0 = completamente irrelevante.
Responde únicamente con el número, sin texto adicional."""

PROMPT_PRECISION = """Dado el contexto recuperado y la pregunta, evalúa qué proporción
del contexto es realmente útil para responder la pregunta.
Pregunta: {pregunta}
Contexto: {contexto}
Responde SOLO con un número decimal entre 0.0 y 1.0 donde:
1.0 = todo el contexto es útil, 0.0 = nada del contexto es útil.
Responde únicamente con el número, sin texto adicional."""

PROMPT_RECALL = """Dada la respuesta de referencia y el contexto recuperado, evalúa
si el contexto contiene toda la información necesaria para construir la respuesta de referencia.
Respuesta de referencia: {referencia}
Contexto recuperado: {contexto}
Responde SOLO con un número decimal entre 0.0 y 1.0 donde:
1.0 = el contexto cubre completamente la referencia, 0.0 = no cubre nada.
Responde únicamente con el número, sin texto adicional."""


def extraer_score(respuesta_llm: str) -> float:
    """Extrae el score numérico de la respuesta del LLM."""
    try:
        texto = respuesta_llm.strip().replace(",", ".")
        import re
        numeros = re.findall(r"\d+\.?\d*", texto)
        if numeros:
            valor = float(numeros[0])
            return min(1.0, max(0.0, valor))
    except Exception:
        pass
    return 0.5


def evaluar_con_llm(prompt: str) -> float:
    """Evalúa una métrica usando el LLM local."""
    respuesta = llm.invoke([
        SystemMessage(content="Eres un evaluador de sistemas RAG. "
                              "Responde ÚNICAMENTE con un número decimal entre 0.0 y 1.0."),
        HumanMessage(content=prompt)
    ])
    return extraer_score(respuesta.content)


# Evaluación par a par
for i, par in enumerate(gt_eval):
    try:
        # Obtener respuesta del sistema RAG
        resultado = consulta_rag(par["pregunta"], k_candidatos=20, k_final=5, verbose=False)
        respuesta_rag = resultado["respuesta_raw"]
        contexto_str  = "\n".join([r["contenido"][:300] for r in resultado["fragmentos"]])

        # Calcular las 4 métricas
        f_score = evaluar_con_llm(PROMPT_FAITHFULNESS.format(
            contexto=contexto_str, respuesta=respuesta_rag))

        r_score = evaluar_con_llm(PROMPT_RELEVANCY.format(
            pregunta=par["pregunta"], respuesta=respuesta_rag))

        p_score = evaluar_con_llm(PROMPT_PRECISION.format(
            pregunta=par["pregunta"], contexto=contexto_str))

        rc_score = evaluar_con_llm(PROMPT_RECALL.format(
            referencia=par["respuesta_referencia"], contexto=contexto_str))

        scores["faithfulness"].append(f_score)
        scores["answer_relevancy"].append(r_score)
        scores["context_precision"].append(p_score)
        scores["context_recall"].append(rc_score)

        if (i + 1) % 5 == 0:
            print(f"  Progreso: {i+1}/{N_EVAL} | "
                  f"Faith: {sum(scores['faithfulness'])/len(scores['faithfulness']):.3f} | "
                  f"Relev: {sum(scores['answer_relevancy'])/len(scores['answer_relevancy']):.3f}")

    except Exception as e:
        logger.warning(f"Error en par {i}: {e}")
        for k in scores:
            scores[k].append(0.5)

# Resultados finales
UMBRALES = {
    "faithfulness"     : 0.85,
    "answer_relevancy" : 0.80,
    "context_precision": 0.75,
    "context_recall"   : 0.75,
}

resultados_metricas = {}
print("\n" + "=" * 60)
print("RESULTADOS — Sistema RAG HybM (BM25 + bge-m3 + RRF) con LLaMA 3.1 8B (Ollama)")
print("=" * 60)
print(f"{'Métrica':<25} {'Valor':>8} {'Umbral':>8} {'Estado':>10}")
print("-" * 60)

for nombre, umbral in UMBRALES.items():
    valores = scores[nombre]
    valor = sum(valores) / len(valores) if valores else 0.0
    estado = "[OK]" if valor >= umbral else "[BAJO]"
    print(f"  {nombre:<23} {valor:>8.4f} {umbral:>8.2f} {estado:>10}")
    resultados_metricas[nombre] = {
        "valor": round(valor, 4),
        "umbral": umbral,
        "aprobado": valor >= umbral,
        "n_eval": len(valores)
    }

n_aprobadas = sum(1 for m in resultados_metricas.values() if m["aprobado"])
print("-" * 60)
print(f"  Métricas superadas : {n_aprobadas}/4")
print(f"  Pares evaluados    : {N_EVAL}")
print("=" * 60)

# Persistencia
with open(GROUND_TRUTH_DIR / "resultados_hyb_bge.json", "w", encoding="utf-8") as f:
    json.dump(resultados_metricas, f, ensure_ascii=False, indent=2)

print(f"\nResultados persistidos en: resultados_hyb_bge.json")

Evaluando 20 pares Q&A...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 15:48:17,892 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:49:00,582 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:49:02,845 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:49:06,094 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:49:09,376 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 15:49:18,355 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:50:44,329 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:50:47,824 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:50:50,817 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 15:50:54,114 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 16:12:55,165 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:14:12,779 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:14:14,563 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:14:16,067 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:14:17,540 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 16:14:21,957 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:15:20,544 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:15:22,077 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:15:24,292 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 16:15:26,834 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


  Progreso: 20/20 | Faith: 0.860 | Relev: 0.825

RESULTADOS — Sistema RAG HybM (BM25 + bge-m3 + RRF) con LLaMA 3.1 8B (Ollama)
Métrica                      Valor   Umbral     Estado
------------------------------------------------------------
  faithfulness              0.8600     0.85       [OK]
  answer_relevancy          0.8250     0.80       [OK]
  context_precision         0.6950     0.75     [BAJO]
  context_recall            0.7950     0.75       [OK]
------------------------------------------------------------
  Métricas superadas : 3/4
  Pares evaluados    : 20

Resultados persistidos en: resultados_hyb_bge.json


---
## Análisis comparativo baseline vs bge-m3 en métricas usando el LLM local

Se repite la evaluación con el sistema híbrido baseline (BM25 + MiniLM)
para comparar cuantitativamente ambas configuraciones.

In [18]:
N_COMP = 10

def evaluar_configuracion(
    nombre: str,
    modelo_emb: SentenceTransformer,
    coleccion: chromadb.Collection,
    gt_pares: list[dict],
    n_eval: int = 10
) -> dict:
    """
    Evalúa una configuración de recuperación sobre el ground truth
    usando el LLM local como juez. Retorna las cuatro métricas promedio.
    """
    scores_cfg = {
        "faithfulness"      : [],
        "answer_relevancy"  : [],
        "context_precision" : [],
        "context_recall"    : [],
    }

    for par in gt_pares[:n_eval]:
        try:
            res_sem  = busqueda_semantica_RAG(par["pregunta"], modelo_emb, coleccion, 20, "es")
            res_bm25 = busqueda_bm25_RAG(par["pregunta"], indice_bm25, fragmentos, 20, "es")
            frags    = reciprocal_rank_fusion([res_sem, res_bm25], fragmentos, top_n=5)

            contexto, _ = formatear_contexto_con_citas(frags)
            user_prompt  = USER_PROMPT_TEMPLATE.format(
                contexto=contexto, consulta=par["pregunta"]
            )
            resp = llm.invoke([
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=user_prompt)
            ])
            respuesta_rag = resp.content
            contexto_str  = "\n".join([f["contenido"][:300] for f in frags])

            scores_cfg["faithfulness"].append(evaluar_con_llm(
                PROMPT_FAITHFULNESS.format(contexto=contexto_str, respuesta=respuesta_rag)))
            scores_cfg["answer_relevancy"].append(evaluar_con_llm(
                PROMPT_RELEVANCY.format(pregunta=par["pregunta"], respuesta=respuesta_rag)))
            scores_cfg["context_precision"].append(evaluar_con_llm(
                PROMPT_PRECISION.format(pregunta=par["pregunta"], contexto=contexto_str)))
            scores_cfg["context_recall"].append(evaluar_con_llm(
                PROMPT_RECALL.format(referencia=par["respuesta_referencia"], contexto=contexto_str)))
        except Exception as e:
            logger.warning(f"Error en evaluación comparativa: {e}")
            for k in scores_cfg:
                scores_cfg[k].append(0.5)

    return {
        "nombre": nombre,
        **{k: round(sum(v) / len(v), 4) if v else 0.0
           for k, v in scores_cfg.items()},
        "n_eval": n_eval
    }


print(f"Evaluando configuración baseline sobre {N_COMP} pares...")
resultado_baseline = evaluar_configuracion(
    "HybB (BM25 + MiniLM-L12)",
    modelo_baseline,
    coleccion_baseline,
    pares_ground_truth,
    N_COMP
)

print(f"Evaluando configuración bge-m3 sobre {N_COMP} pares...")
resultado_bge_m3_comp = evaluar_configuracion(
    "HybM (BM25 + bge-m3)",
    modelo_bge_m3,
    coleccion_bge_m3,
    pares_ground_truth,
    N_COMP
)

# Tabla comparativa
metricas_comp = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
comparativa   = {}

print("\n" + "=" * 70)
print("COMPARATIVA — Baseline vs bge-m3")
print("=" * 70)
print(f"{'Métrica':<25} {'HybB (MiniLM)':>15} {'HybM (bge-m3)':>15} "
      f"{'Umbral':>8} {'Ganador':>10}")
print("-" * 70)

for metrica in metricas_comp:
    v_base = resultado_baseline[metrica]
    v_bge  = resultado_bge_m3_comp[metrica]
    umbral = UMBRALES[metrica]
    ganador = "HybM" if v_bge > v_base else "HybB" if v_base > v_bge else "Empate"
    print(f"  {metrica:<23} {v_base:>15.4f} {v_bge:>15.4f} {umbral:>8.2f} {ganador:>10}")
    comparativa[metrica] = {
        "baseline": v_base, "bge_m3": v_bge, "umbral": umbral, "ganador": ganador
    }

print("=" * 70)

with open(GROUND_TRUTH_DIR / "comparativa.json", "w", encoding="utf-8") as f:
    json.dump(comparativa, f, ensure_ascii=False, indent=2)

print("Comparativa persistida en: comparativa.json")

Evaluando configuración baseline sobre 10 pares...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 19:13:08,337 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:15:35,520 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:15:38,091 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:15:40,903 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:15:43,125 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 19:15:47,789 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:17:09,134 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:17:11,194 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:17:12,900 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:17:15,062 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 19:45:52,078 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:46:56,571 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:46:58,932 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:47:01,264 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:47:03,635 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-13 19:47:08,952 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:48:26,880 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:48:29,210 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:48:32,441 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-13 19:48:36,139 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



COMPARATIVA — Baseline vs bge-m3
Métrica                     HybB (MiniLM)   HybM (bge-m3)   Umbral    Ganador
----------------------------------------------------------------------
  faithfulness                     0.8600          0.8200     0.85       HybB
  answer_relevancy                 0.8300          0.8100     0.80       HybB
  context_precision                0.6300          0.7100     0.75       HybM
  context_recall                   0.7400          0.7700     0.75       HybM
Comparativa persistida en: comparativa.json


---
## Función de consulta interactiva por celda (streaming)

Se implementa una función de consulta interactiva con streaming de tokens
que simula la experiencia de uso de sistemas como ChatGPT o Gemini.

In [20]:
def consulta_interactiva(consulta: str, k_final: int = 5) -> None:
    """
    Ejecuta una consulta RAG con streaming de tokens en el notebook.
    Muestra la respuesta carácter a carácter simulando la experiencia
    de los asistentes conversacionales modernos.

    Parámetros
    ----------
    consulta : str
        Pregunta en español sobre el AI Act.
    k_final : int
        Número de fragmentos a recuperar e inyectar en el contexto.
    """
    print("\n" + "═" * 65)
    print(" SISTEMA RAG — AUDITORÍA DEL AI ACT")
    print("═" * 65)
    print(f" Consulta: {consulta}")
    print("─" * 65)

    # RECUPERACIÓN HÍBRIDA
    print(" Recuperando fragmentos normativos...", end="", flush=True)
    t0 = time.time()

    res_sem  = busqueda_semantica_RAG(consulta, modelo_bge_m3, coleccion_bge_m3, 20, "es")
    res_bm25 = busqueda_bm25_RAG(consulta, indice_bm25, fragmentos, 20, "es")
    frags    = reciprocal_rank_fusion([res_sem, res_bm25], fragmentos, top_n=k_final)

    t_rec = time.time() - t0
    print(f" {len(frags)} fragmentos recuperados ({t_rec:.2f}s)")

    # Mostrar fuentes recuperadas
    print("\n Fuentes recuperadas:")
    for r in frags:
        cat  = r['metadatos'].get('categoria_fuente', r['metadatos'].get('source', 'N/D'))
        arts = r['metadatos'].get('articulos', '')
        arts_str = f" | Arts: {arts}" if arts else ""
        print(f"  [{r['ranking']}] {str(cat)[:50]}{arts_str}")

    # FORMATEO DEL CONTEXTO
    contexto, referencias = formatear_contexto_con_citas(frags)
    user_prompt = USER_PROMPT_TEMPLATE.format(contexto=contexto, consulta=consulta)

    # STREAMING DE LA RESPUESTA
    print("\n" + "─" * 65)
    print(" Respuesta:\n")

    respuesta_completa = ""
    t0 = time.time()

    for chunk in llm.stream([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]):
        token = chunk.content
        print(token, end="", flush=True)
        respuesta_completa += token

    t_gen = time.time() - t0

    # CITACIONES AL PIE
    print("\n")
    print(formatear_citaciones(referencias))

    print("\n" + "─" * 65)
    print(f" Tiempo recuperación : {t_rec:.2f}s")
    print(f" Tiempo generación   : {t_gen:.2f}s")
    print(f" Tiempo total        : {t_rec + t_gen:.2f}s")
    print("═" * 65)

# VARIABLE PARA CONSULTAR
MI_CONSULTA = "¿Cuáles son los requisitos que debe cumplir un sistema de IA de alto riesgo antes de ser puesto en el mercado europeo?"

consulta_interactiva(MI_CONSULTA)


═════════════════════════════════════════════════════════════════
 SISTEMA RAG — AUDITORÍA DEL AI ACT
═════════════════════════════════════════════════════════════════
 Consulta: ¿Cuáles son los requisitos que debe cumplir un sistema de IA de alto riesgo antes de ser puesto en el mercado europeo?
─────────────────────────────────────────────────────────────────
 Recuperando fragmentos normativos...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 5 fragmentos recuperados (0.45s)

 Fuentes recuperadas:
  [1] AESIA
  [2] AESIA
  [3] Reglamento (UE) 20241689 (AI Act)
  [4] AESIA | Arts: ['9']
  [5] AESIA

─────────────────────────────────────────────────────────────────
 Respuesta:



2026-06-13 19:55:53,053 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Respuesta directa:
Un sistema de IA de alto riesgo debe cumplir con los requisitos establecidos en el Reglamento (UE) 2024/1689, específicamente en lo referente a la gestión de riesgos, antes de ser puesto en el mercado europeo.

Desarrollo:
Según el artículo 9 del Reglamento Europeo de Inteligencia Artificial (Reglamento 2024/1689), los sistemas de IA de alto riesgo deben implementar y mantener un enfoque iterativo y continuo para la gestión de riesgos. Esto implica que el sistema debe ser capaz de identificar, evaluar y mitigar los riesgos asociados con su funcionamiento.

Además, según el Reglamento (UE) 2024/1689, se establece que los sistemas de IA de alto riesgo deben cumplir con determinados requisitos obligatorios, entre los cuales está la gestión de riesgos. Estos requisitos tienen como objetivo garantizar que los sistemas de IA de alto riesgo disponibles en la Unión o cuyos resultados de salida se utilicen en la Unión no representen riesgos inaceptables para intereses público

---
## Resumen de Fase 3 y persistencia del estado

In [21]:
estado_fase3 = {
    "modelo_generativo": {
        "id"          : MODELO_LLM_ID,
        "backend"     : "Ollama",
        "base_url"    : OLLAMA_BASE_URL,
        "temperatura" : 0.0,
        "num_predict" : 1024,
        "num_ctx"     : 4096
    },
    "recuperador": {
        "tipo"         : "hibrido_rrf",
        "embedding"    : estado_fase2["modelos"]["bge_m3"]["id"],
        "lexical"      : "BM25",
        "k_candidatos" : 20,
        "k_final"      : 5,
        "k_rrf"        : 60,
        "filtro_idioma": "es"
    },
    "componentes": [
        "Etiquetado de idioma en ChromaDB",
        "Stopwords ampliadas para dominio jurídico",
        "Normalización de acentos en tokenizador BM25",
        "Persistencia del índice BM25"
    ],
    "evaluacion": resultados_metricas,
    "ground_truth": {
        "n_pares" : len(pares_ground_truth),
        "fichero" : str(RUTA_GROUND_TRUTH)
    },
    "artefactos": {
        "bm25_pkl"    : str(RUTA_BM25),
        "ground_truth": str(RUTA_GROUND_TRUTH),
        "resultados"  : str(GROUND_TRUTH_DIR / "resultados_hyb_bge.json"),
        "comparativa" : str(GROUND_TRUTH_DIR / "comparativa.json")
    }
}

RUTA_ESTADO_FASE3 = BASE_DIR / "fase3_estado.json"
with open(RUTA_ESTADO_FASE3, "w", encoding="utf-8") as f:
    json.dump(estado_fase3, f, ensure_ascii=False, indent=2)

print("=" * 65)
print("RESUMEN — ORQUESTACIÓN GENERATIVA")
print("=" * 65)
print(f"  Modelo generativo  : {MODELO_LLM_ID} (Ollama, temperatura=0.0)")
print(f"  Recuperador activo : HybM (BM25 + bge-m3 + RRF)")
print(f"  Filtro de idioma   : español (es)")
print(f"  Ground truth       : {len(pares_ground_truth)} pares Q&A")
print()
print("  Métricas de evaluación (sistema HybM):")
for nombre, datos in resultados_metricas.items():
    estado = "OK" if datos["aprobado"] else "BAJO"
    print(f"    {nombre:<25}: {datos['valor']:.4f} [{estado}]")
print()
print("  Artefactos generados:")
print(f"    - {RUTA_BM25.name}")
print(f"    - {RUTA_GROUND_TRUTH.name}")
print(f"    - resultados_hyb_bge.json")
print(f"    - comparativa.json")
print(f"    - fase3_estado.json")
print("=" * 65)

RESUMEN — ORQUESTACIÓN GENERATIVA
  Modelo generativo  : llama3.1:8b (Ollama, temperatura=0.0)
  Recuperador activo : HybM (BM25 + bge-m3 + RRF)
  Filtro de idioma   : español (es)
  Ground truth       : 60 pares Q&A

  Métricas de evaluación (sistema HybM):
    faithfulness             : 0.8600 [OK]
    answer_relevancy         : 0.8250 [OK]
    context_precision        : 0.6950 [BAJO]
    context_recall           : 0.7950 [OK]

  Artefactos generados:
    - indice_bm25.pkl
    - ground_truth_ai_act.json
    - resultados_hyb_bge.json
    - comparativa.json
    - fase3_estado.json
